# Markdown → Google Doc (Google Colab)

This notebook authenticates with Google, creates a new Google Doc, and converts the provided markdown meeting notes into a formatted document (headings, nested bullets, checkboxes, styled @mentions, and styled footer).

In [ ]:
# ✅ 1) Install dependencies (Colab usually already has these, but this is safe)
!pip -q install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

# ✅ 2) Markdown source (replace this string or read from a .md file)
MARKDOWN_NOTES = '# Product Team Sync - May 15, 2023\n\n## Attendees\n- Sarah Chen (Product Lead)\n- Mike Johnson (Engineering)\n- Anna Smith (Design)\n- David Park (QA)\n\n## Agenda\n\n### 1. Sprint Review\n* Completed Features\n  * User authentication flow\n  * Dashboard redesign\n  * Performance optimization\n    * Reduced load time by 40%\n    * Implemented caching solution\n* Pending Items\n  * Mobile responsive fixes\n  * Beta testing feedback integration\n\n### 2. Current Challenges\n* Resource constraints in QA team\n* Third-party API integration delays\n* User feedback on new UI\n  * Navigation confusion\n  * Color contrast issues\n\n### 3. Next Sprint Planning\n* Priority Features\n  * Payment gateway integration\n  * User profile enhancement\n  * Analytics dashboard\n* Technical Debt\n  * Code refactoring\n  * Documentation updates\n\n## Action Items\n- [ ] @sarah: Finalize Q3 roadmap by Friday\n- [ ] @mike: Schedule technical review for payment integration\n- [ ] @anna: Share updated design system documentation\n- [ ] @david: Prepare QA resource allocation proposal\n\n## Next Steps\n* Schedule individual team reviews\n* Update sprint board\n* Share meeting summary with stakeholders\n\n## Notes\n* Next sync scheduled for May 22, 2023\n* Platform demo for stakeholders on May 25\n* Remember to update JIRA tickets\n\n---\nMeeting recorded by: Sarah Chen\nDuration: 45 minutes\n'

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any

from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


# -----------------------------
# Parsing utilities
# -----------------------------

@dataclass
class Block:
    """Represents a single line/block to be written to the Google Doc."""
    text: str
    kind: str  # "h1", "h2", "h3", "p", "bullet", "checkbox", "hr", "footer"
    level: int = 0  # nesting level for bullets / checkboxes


HEADING_MAP = {"h1": "HEADING_1", "h2": "HEADING_2", "h3": "HEADING_3"}
MENTION_RE = re.compile(r"(@[a-zA-Z0-9_]+)")


def _indent_level(line: str) -> int:
    """Determine nesting level based on leading spaces (2 spaces per level)."""
    expanded = line.replace("\t", "  ")
    leading = len(expanded) - len(expanded.lstrip(" "))
    return max(0, leading // 2)


def parse_markdown_to_blocks(md: str) -> List[Block]:
    """Parse a limited markdown subset tailored to the meeting notes."""
    lines = md.splitlines()
    blocks: List[Block] = []
    in_footer = False

    for raw in lines:
        line = raw.rstrip("\n")
        stripped = line.strip()

        if not stripped:
            blocks.append(Block(text="", kind="p"))
            continue

        if stripped == "---":
            blocks.append(Block(text="", kind="hr"))
            in_footer = True
            continue

        if in_footer:
            blocks.append(Block(text=stripped, kind="footer"))
            continue

        if stripped.startswith("# "):
            blocks.append(Block(text=stripped[2:].strip(), kind="h1"))
            continue
        if stripped.startswith("## "):
            blocks.append(Block(text=stripped[3:].strip(), kind="h2"))
            continue
        if stripped.startswith("### "):
            blocks.append(Block(text=stripped[4:].strip(), kind="h3"))
            continue

        # Checkboxes: - [ ] text
        m_cb = re.match(r"^(\s*)-\s+\[\s*\]\s+(.*)$", line)
        if m_cb:
            lvl = _indent_level(m_cb.group(1))
            blocks.append(Block(text=m_cb.group(2).strip(), kind="checkbox", level=lvl))
            continue

        # Bullets: - text OR * text (with indentation)
        m_b = re.match(r"^(\s*)[-*]\s+(.*)$", line)
        if m_b:
            lvl = _indent_level(m_b.group(1))
            blocks.append(Block(text=m_b.group(2).strip(), kind="bullet", level=lvl))
            continue

        blocks.append(Block(text=stripped, kind="p"))

    return blocks


def find_mentions(text: str) -> List[Tuple[int, int]]:
    """Return spans for @mentions in a line of text."""
    return [(m.start(), m.end()) for m in MENTION_RE.finditer(text)]


# -----------------------------
# Google Docs integration
# -----------------------------

def get_docs_service():
    """Authenticate in Colab and return a Docs API service client."""
    try:
        auth.authenticate_user()
        creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/documents"])
        return build("docs", "v1", credentials=creds)
    except Exception as e:
        raise RuntimeError(f"Authentication/setup failed: {e}")


def create_document(service, title: str) -> str:
    """Create a new Google Doc and return its documentId."""
    try:
        doc = service.documents().create(body={"title": title}).execute()
        return doc["documentId"]
    except HttpError as e:
        raise RuntimeError(f"Docs API error while creating document: {e}")


def _build_insert_and_ranges(blocks: List[Block]):
    """Insert all text in one request and compute paragraph ranges for styling."""
    parts: List[str] = []
    ranges: List[Tuple[int, int]] = []

    cursor = 1  # Doc body starts at index 1
    for b in blocks:
        chunk = (b.text + "\n")
        parts.append(chunk)
        start = cursor
        end = cursor + len(chunk)
        ranges.append((start, end))
        cursor = end

    full_text = "".join(parts)

    insert_requests = [{
        "insertText": {"location": {"index": 1}, "text": full_text}
    }]
    return insert_requests, ranges


def build_formatting_requests(blocks: List[Block], ranges: List[Tuple[int, int]]):
    """Create batchUpdate requests to apply headings, bullets, checkboxes, mention/footer styling."""
    requests: List[Dict[str, Any]] = []
    bullet_ranges_by_level: Dict[int, List[Tuple[int, int]]] = {}
    checkbox_ranges_by_level: Dict[int, List[Tuple[int, int]]] = {}
    footer_ranges: List[Tuple[int, int]] = []

    for b, (start, end) in zip(blocks, ranges):
        if b.kind in ("h1", "h2", "h3"):
            requests.append({
                "updateParagraphStyle": {
                    "range": {"startIndex": start, "endIndex": end},
                    "paragraphStyle": {"namedStyleType": HEADING_MAP[b.kind]},
                    "fields": "namedStyleType",
                }
            })

        if b.kind == "bullet":
            bullet_ranges_by_level.setdefault(b.level, []).append((start, end))

        if b.kind == "checkbox":
            checkbox_ranges_by_level.setdefault(b.level, []).append((start, end))

        if b.kind == "footer":
            footer_ranges.append((start, end))

        # Style @mentions
        for ms, me in find_mentions(b.text):
            requests.append({
                "updateTextStyle": {
                    "range": {"startIndex": start + ms, "endIndex": start + me},
                    "textStyle": {
                        "bold": True,
                        "foregroundColor": {"color": {"rgbColor": {"red": 0.10, "green": 0.35, "blue": 0.95}}},
                    },
                    "fields": "bold,foregroundColor",
                }
            })

    # Apply bullet lists with nesting
    for lvl, lvl_ranges in bullet_ranges_by_level.items():
        for s, e in lvl_ranges:
            requests.append({
                "createParagraphBullets": {
                    "range": {"startIndex": s, "endIndex": e},
                    "bulletPreset": "BULLET_DISC_CIRCLE_SQUARE",
                    "nestingLevel": lvl,
                }
            })

    # Apply checkboxes with nesting
    for lvl, lvl_ranges in checkbox_ranges_by_level.items():
        for s, e in lvl_ranges:
            requests.append({
                "createParagraphBullets": {
                    "range": {"startIndex": s, "endIndex": e},
                    "bulletPreset": "BULLET_CHECKBOX",
                    "nestingLevel": lvl,
                }
            })

    # Footer styling
    for s, e in footer_ranges:
        requests.append({
            "updateTextStyle": {
                "range": {"startIndex": s, "endIndex": e},
                "textStyle": {
                    "italic": True,
                    "fontSize": {"magnitude": 10, "unit": "PT"},
                    "foregroundColor": {"color": {"rgbColor": {"red": 0.40, "green": 0.40, "blue": 0.40}}},
                },
                "fields": "italic,fontSize,foregroundColor",
            }
        })

    return requests


def write_markdown_to_gdoc(md: str, doc_title: str = "Product Team Sync") -> str:
    """Create a Google Doc and write formatted content from markdown."""
    blocks = parse_markdown_to_blocks(md)
    service = get_docs_service()
    document_id = create_document(service, doc_title)

    insert_requests, ranges = _build_insert_and_ranges(blocks)
    formatting_requests = build_formatting_requests(blocks, ranges)

    try:
        service.documents().batchUpdate(documentId=document_id, body={"requests": insert_requests}).execute()
        service.documents().batchUpdate(documentId=document_id, body={"requests": formatting_requests}).execute()
    except HttpError as e:
        raise RuntimeError(f"Docs API error during batchUpdate: {e}")
    except Exception as e:
        raise RuntimeError(f"Unexpected error while writing the document: {e}")

    return document_id

In [ ]:
# ✅ Run the conversion
try:
    doc_id = write_markdown_to_gdoc(MARKDOWN_NOTES, doc_title="Product Team Sync")
    print("✅ Created Google Doc!")
    print("Document ID:", doc_id)
    print(f"Open it: https://docs.google.com/document/d/{doc_id}/edit")
except Exception as e:
    print("❌ Failed:", e)